# SCT Evaluation

Evaluates LLM responses on Script Concordance Test (SCT) questions against expert distributions.

**Outputs:** SCT scores, rating distributions, calibration curves, ROC curves, strategy analysis, response entropy.

In [ ]:
import os
import pandas as pd
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import datetime
import time
import re
import logging
from collections import Counter
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_PATH = '.'
REASON = False # Set to True if using reasoning folder
COMPARE = True  # set to True to compare across model variants
if REASON:
    INPUT_PATH = BASE_PATH + '/data/output/sct_w_reason_zero-shot'
    EVAL_PATH = BASE_PATH + '/data/evaluation/sct_w_reason_zero-shot'
elif COMPARE:
    INPUT_PATH = BASE_PATH + '/data/output/gpt4-1_comparison'
    EVAL_PATH = BASE_PATH + '/data/evaluation/gpt4-1_comparison/V2'
else:
    INPUT_PATH = BASE_PATH + '/data/output/sct_wo_reason_zero-shot'
    EVAL_PATH = BASE_PATH + '/data/evaluation/sct_wo_reason_zero-shot'

os.makedirs(EVAL_PATH, exist_ok=True)

# Load expert distributions CSV
EXPERT_CSV_PATH = './data/sct_cleaned_full.csv'  # Update path as needed

In [ ]:
# =================================================================
# GLOBAL LABEL MAPPING FOR ALL PLOTS
# =================================================================

def create_clean_labels(file_names):
    """
    Convert long filenames to clean, presentation-ready labels.
    Uses your script's REASON and COMPARE flags for context-aware labeling.
    """
    label_mapping = {}
    
    for name in file_names:
        # Convert to lowercase for pattern matching
        name_lower = name.lower()
        
        if COMPARE:
            if "reason" in name_lower:
                clean_name = "GPT-4.1 CoT"
            elif "base" in name_lower:
                clean_name = "GPT-4.1 Base"
            else:
                # Fallback for other files in comparison mode
                clean_name = "GPT-4.1"
        
        elif REASON:
            # Extract model name and add CoT
            if "gpt-4.1" in name_lower or "gpt4-1" in name_lower:
                clean_name = "GPT-4.1 CoT"
            elif "gpt-4" in name_lower:
                clean_name = "GPT-4 CoT"
            elif "claude" in name_lower:
                clean_name = "Claude CoT"
            elif "llama" in name_lower:
                clean_name = "Llama CoT"
            elif "openai" in name_lower:
                clean_name = "OpenAI CoT"
            else:
                # Fallback: try to extract model name from filename
                # Remove common prefixes/suffixes and add CoT
                clean_base = name.replace("responses_", "").replace("_reason", "").replace("_cleaned", "")
                clean_name = f"{clean_base[:10]} CoT" if len(clean_base) > 10 else f"{clean_base} CoT"
        
        else:
            # Extract simple model names
            if "gpt-4.1" in name_lower or "gpt4-1" in name_lower:
                clean_name = "GPT-4.1"
            elif "gpt-4" in name_lower:
                clean_name = "GPT-4"
            elif "claude" in name_lower:
                clean_name = "Claude"
            elif "llama" in name_lower:
                clean_name = "Llama"
            elif "openai" in name_lower:
                clean_name = "OpenAI"
            else:
                # Fallback: abbreviate long names
                clean_name = name[:15] + "..." if len(name) > 15 else name
        
        label_mapping[name] = clean_name
    
    return label_mapping

def get_clean_labels(file_names):
    """
    Simple wrapper function that returns clean labels for any list of filenames.
    Use this in all your plotting functions.
    """
    if isinstance(file_names, list):
        label_mapping = create_clean_labels(file_names)
        return [label_mapping[name] for name in file_names]
    else:
        # Handle single filename
        label_mapping = create_clean_labels([file_names])
        return label_mapping[file_names]

# Instruction for calling the label functions in plots:
# clean_labels = get_clean_labels(file_names)
# plt.xticks(range(len(file_names)), clean_labels, rotation=45, ha='right')

In [ ]:
# ============================================================================
# DATA LOADING
# ============================================================================

response_files = glob.glob(f'{INPUT_PATH}/responses_*.json')
response_files.sort(key=os.path.getmtime, reverse=True)

response_dfs = []
for filepath in response_files:
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            df = pd.DataFrame(data)
            df['source_file'] = os.path.basename(filepath)
            response_dfs.append(df)
            print(f"Loaded {len(df)} items from {os.path.basename(filepath)}")
    except Exception as e:
        print(f"Error loading {filepath}: {e}")

print(f"\nFound {len(response_dfs)} response files")
if response_dfs:
    print(f"Sample columns from first file: {response_dfs[0].columns.tolist()}")

os.makedirs(EVAL_PATH, exist_ok=True)
os.makedirs(os.path.join(EVAL_PATH, 'plots'), exist_ok=True)

In [ ]:
# ============================================================================
# RESPONSE PROCESSING FUNCTIONS
# ============================================================================
def process_sct_response(row, error_file):
    """
    Optimized SCT response processing for asterisk-free text.
    Handles variations in case, spacing, and punctuation for rating, uncertainty, and strategy extraction.
    
    Args:
        row: Single row from response DataFrame
        error_file: File handle for logging errors
        
    Returns:
        Dictionary with processed response data
    """
    response_data = row.get("response", {})
    question_id = int(row.get("question_id"))
    question = row.get("question", "")
    sct_stem = row.get("sct_stem", "")
    additional_info = row.get("additional_info", "")

    # Get expert distribution directly from response file row
    expert_cols = ['-2', '-1', '0', '1', '2']
    try:
        expert_dist = np.array([float(row.get(col, 0)) for col in expert_cols])
        
        # Validate expert distribution
        if expert_dist.sum() == 0:  # Check if all zeros
            error_file.write(json.dumps({
                "question_id": question_id,
                "error": "Expert distribution is all zeros"
            }) + "\n")
            return None
            
        # Check for any NaN values
        if np.isnan(expert_dist).any():
            error_file.write(json.dumps({
                "question_id": question_id,
                "error": "Expert distribution contains NaN values"
            }) + "\n")
            return None
            
    except (ValueError, TypeError) as e:
        error_file.write(json.dumps({
            "question_id": question_id,
            "error": f"Could not extract expert distribution: {e}",
            "expert_values": {col: row.get(col, "missing") for col in expert_cols}
        }) + "\n")
        return None
    
    # Normalize expert distribution (max value gets score of 1.0)
    if expert_dist.max() > 0:
        expert_dist_normalized = expert_dist / expert_dist.max()
    else:
        expert_dist_normalized = expert_dist

    results_all = {}
    
    try:
        for key, response_value in response_data.items():
            try:
                # Extract text and remove asterisks
                resp_text = str(response_value)
                reasoning_tok = None  # No reasoning tokens in SCT format
                
                # *** REMOVE ASTERISKS ***
                resp_text = resp_text.replace('*', '')
                
                # ================================================================
                # RATING EXTRACTION: "rating:" keyword only
                # ================================================================
                rating = None
                score = None
                
                # Look for "rating:" with flexible spacing and case
                # Handles: "rating: -2", "rating : +1", "Rating: 0", "rating:-2", etc.
                rating_match = re.search(r'\brating\s*:\s*([+-]?[012])\b', resp_text, re.IGNORECASE)
                
                if rating_match:
                    rating_str = rating_match.group(1).replace(' ', '')  # Remove any spaces
                    # Handle the + sign explicitly
                    if rating_str.startswith('+'):
                        rating = int(rating_str[1:])
                    elif rating_str.startswith('-'):
                        rating = int(rating_str)  # Keep the negative sign
                    else:
                        rating = int(rating_str)
                    
                    # Validate rating is in range [-2, +2]
                    if -2 <= rating <= 2:
                        # Calculate score based on expert distribution
                        rating_idx = rating + 2  # Convert -2,+2 range to 0,4 index
                        score = expert_dist_normalized[rating_idx]
                    else:
                        # Log out-of-range but still include the response
                        error_file.write(json.dumps({
                            "question_id": question_id,
                            "response_number": key,
                            "error_type": "rating_out_of_range",
                            "extracted_value": rating,
                            "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                        }) + "\n")
                        rating = None
                        score = None

                # ================================================================
                # UNCERTAINTY EXTRACTION: "uncertainty:" keyword only
                # ================================================================
                uncertainty = None
                
                if rating is not None:  # Only extract uncertainty if we found a rating
                    # Look for "uncertainty:" with flexible spacing and case
                    # Handles: "uncertainty: 90", "uncertainty : 85", "Uncertainty: 75", "uncertainty:90", etc.
                    uncertainty_match = re.search(r'\buncertainty\s*:\s*(\d{1,3})\b', resp_text, re.IGNORECASE)
                    
                    if uncertainty_match:
                        try:
                            temp_uncertainty = int(uncertainty_match.group(1))
                            if 0 <= temp_uncertainty <= 100:
                                uncertainty = temp_uncertainty
                            else:
                                # Log out-of-range but still include the response
                                error_file.write(json.dumps({
                                    "question_id": question_id,
                                    "response_number": key,
                                    "error_type": "uncertainty_out_of_range",
                                    "extracted_value": temp_uncertainty,
                                    "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                                }) + "\n")
                        except ValueError:
                            # Log parsing error but still include the response
                            error_file.write(json.dumps({
                                "question_id": question_id,
                                "response_number": key,
                                "error_type": "uncertainty_parsing_error",
                                "raw_match": uncertainty_match.group(1) if uncertainty_match else "None",
                                "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                            }) + "\n")
                    else:
                        # Log uncertainty extraction failure but still include the response
                        error_file.write(json.dumps({
                            "question_id": question_id,
                            "response_number": key,
                            "error_type": "uncertainty_extraction_failed",
                            "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                        }) + "\n")

                # ================================================================
                # STRATEGY EXTRACTION: Auto-detect if both keywords present
                # ================================================================
                strategies = []
                
                # Auto-detect reasoning content: only extract if both keywords are present
                has_strategy_keyword = re.search(r'\bstrategy\s*:', resp_text, re.IGNORECASE)
                has_justification_keyword = re.search(r'\bjustification\s*:', resp_text, re.IGNORECASE)
                
                if has_strategy_keyword and has_justification_keyword and rating is not None:
                    # Look for text between "strategy:" and "justification:" with flexible spacing and case
                    # Handles: "strategy: text justification:", "Strategy : text Justification:", etc.
                    strategy_match = re.search(r'\bstrategy\s*:\s*(.*?)\s*\bjustification\s*:', resp_text, re.IGNORECASE | re.DOTALL)
                    
                    if strategy_match:
                        strategy_text = strategy_match.group(1).strip()
                        # Split by commas and clean up
                        for strategy in strategy_text.split(','):
                            strategy = strategy.strip()
                            # Remove any remaining asterisks and normalize whitespace
                            strategy = re.sub(r'\*+', '', strategy).strip()
                            strategy = re.sub(r'\s+', ' ', strategy)  # Normalize whitespace
                            if strategy and len(strategy) > 2:  # Filter out very short strings
                                strategies.append(strategy)

                # ================================================================
                # INCLUDE RESPONSE IF RATING FOUND
                # ================================================================
                if rating is not None and score is not None:
                    results_all[key] = {
                        "rating": rating,
                        "score": float(score),
                        "uncertainty": uncertainty,  # Can be None
                        "strategies": strategies,
                        "reasoning_tokens": reasoning_tok,
                        "expert_distribution": expert_dist.tolist(),
                        "expert_modal_response": int(np.argmax(expert_dist) - 2)  # Convert back to -2,+2
                    }
                else:
                    # Log rating extraction failure
                    error_file.write(json.dumps({
                        "question_id": question_id,
                        "response_number": key,
                        "error_type": "rating_extraction_failed",
                        "pattern_used": "\\brating\\s*:\\s*([+-]?[012])\\b",
                        "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                    }) + "\n")
                        
            except Exception as e:
                # Log processing exception but continue
                error_file.write(json.dumps({
                    "question_id": question_id,
                    "response_number": key,
                    "error_type": "processing_exception",
                    "error_message": str(e),
                    "response_preview": str(response_value)[:200] + "..." if len(str(response_value)) > 200 else str(response_value)
                }) + "\n")
                continue
        
        return {
            "question_id": question_id,
            "question": question,
            "sct_stem": sct_stem,
            "additional_info": additional_info,
            "expert_distribution": expert_dist.tolist(),
            "expert_modal_response": int(np.argmax(expert_dist) - 2),
            "response": response_data,
            "results": results_all
        }
        
    except Exception as e:
        # Log fatal error
        error_file.write(json.dumps({
            "question_id": question_id,
            "error_type": "fatal_processing_error",
            "error_message": str(e)
        }) + "\n")
        
    return None

In [ ]:
processed_data = {}
sct_scores = {}

for df in response_dfs:
    filename = df['source_file'].iloc[0]
    base_name = os.path.splitext(filename)[0]
    
    print(f"\nProcessing {filename}...")
    
    error_filename = f"errors_{base_name}.jsonl"
    error_path = os.path.join(EVAL_PATH, error_filename)
    
    with open(error_path, "w") as error_file:
        results = df.apply(lambda row: process_sct_response(row, error_file), axis=1)
    
    df_eval = pd.DataFrame([r for r in results.tolist() if r is not None])
    df_eval['source_file'] = filename
    
    print(f"  Successfully processed: {len(df_eval)}/{len(df)} questions")
    
    processed_data[base_name] = df_eval

print(f"\nProcessed {len(processed_data)} models successfully")

In [ ]:
# ============================================================================
# METRICS CALCULATION FUNCTIONS - NEW VERSION WITH INDIVIDUAL RESPONSES
# ============================================================================

import numpy as np
from collections import Counter

def calculate_sct_metrics(row):
    """
    Calculate SCT metrics for each question focusing on the most frequent rating.
    
    Returns:
    - most_frequent_rating: The rating chosen most often
    - most_frequent_score: The score for that rating (not averaged)
    - rating_distribution: Count of each rating {-2: 2, -1: 5, 0: 3, 1: 4, 2: 1}
    - rating_distribution_normalized: Weights relative to most frequent (most frequent = 1.0)
    - most_frequent_uncertainty: Average uncertainty of responses with most frequent rating
    - self_consistency: Fraction of responses that chose most frequent rating
    - expert_modal_agreement: True if most frequent rating matches expert consensus
    - individual_responses: Dictionary storing each response's rating and uncertainty
    """
    results_all = row['results']
    total_responses = len(results_all)
    
    if total_responses == 0:
        return None
    
    # Get expert distribution and find expert consensus (rating with weight 1.0)
    expert_dist = np.array(row['expert_distribution'])
    expert_consensus_idx = np.argmax(expert_dist)  # Index of highest weight
    expert_consensus_rating = expert_consensus_idx - 2  # Convert back to -2,+2 scale
    
    # NEW: Store individual response data
    individual_responses = {}
    
    # Collect ratings and group by rating
    ratings = []
    responses_by_rating = {}  # {rating: [list of response data]}
    
    for key, result in results_all.items():
        rating = result['rating']
        uncertainty = result['uncertainty']
        ratings.append(rating)
        
        # NEW: Store individual response data
        individual_responses[key] = {
            'rating': rating,
            'uncertainty': uncertainty
        }
        
        if rating not in responses_by_rating:
            responses_by_rating[rating] = []
        responses_by_rating[rating].append(result)
    
    # Find most frequent rating
    rating_counts = Counter(ratings)
    most_frequent_rating = rating_counts.most_common(1)[0][0]
    most_frequent_count = rating_counts.most_common(1)[0][1]
    
    # Get score for most frequent rating (just one score, not averaged)
    # All responses with the same rating should have the same score
    most_frequent_score = responses_by_rating[most_frequent_rating][0]['score']
    
    # Create normalized rating distribution (most frequent = 1.0)
    rating_distribution = dict(rating_counts)
    rating_distribution_normalized = {}
    for rating, count in rating_distribution.items():
        rating_distribution_normalized[rating] = count / most_frequent_count
    
    # NEW: Create self-consistency dictionary for each rating (like MedQA format)
    self_consistency = {}
    for rating in [-2, -1, 0, 1, 2]:
        count = rating_counts.get(rating, 0)
        self_consistency[rating] = count / total_responses
    
    # Calculate most frequent uncertainty (average of uncertainties for most frequent rating only)
    most_frequent_uncertainties = []
    for result in responses_by_rating[most_frequent_rating]:
        if result['uncertainty'] is not None:
            most_frequent_uncertainties.append(result['uncertainty'])
    
    most_frequent_uncertainty = np.mean(most_frequent_uncertainties) if most_frequent_uncertainties else None
    
    # Self-consistency for most frequent rating (fraction of responses that chose most frequent rating)
    most_frequent_self_consistency = most_frequent_count / total_responses
    
    # Expert modal agreement (most frequent rating matches expert consensus)
    expert_modal_agreement = bool(most_frequent_rating == expert_consensus_rating)
    
    # Strategy metrics (if available)
    all_strategies = []
    reasoning_tokens = []
    for result in results_all.values():
        if 'strategies' in result and result['strategies']:
            all_strategies.extend(result['strategies'])
        if result['reasoning_tokens'] is not None:
            reasoning_tokens.append(result['reasoning_tokens'])
    
    strategy_counts = Counter(all_strategies) if all_strategies else Counter()
    avg_reasoning_tokens = np.mean(reasoning_tokens) if reasoning_tokens else None
    
    return {
        "total_responses": int(total_responses),
        "most_frequent_rating": int(most_frequent_rating),
        "most_frequent_score": float(most_frequent_score),
        "rating_distribution": {int(k): int(v) for k, v in rating_distribution.items()},
        "rating_distribution_normalized": {int(k): float(v) for k, v in rating_distribution_normalized.items()},
        "most_frequent_uncertainty": float(most_frequent_uncertainty) if most_frequent_uncertainty is not None else None,
        "self_consistency": {int(k): float(v) for k, v in self_consistency.items()},  # NEW: Per-rating self-consistency
        "most_frequent_self_consistency": float(most_frequent_self_consistency),  # Keep original for compatibility
        "expert_modal_agreement": expert_modal_agreement,
        "expert_consensus_rating": int(expert_consensus_rating),
        "average_reasoning_tokens": float(avg_reasoning_tokens) if avg_reasoning_tokens is not None else None,
        "strategies": dict(strategy_counts),
        "unique_strategies": int(len(strategy_counts)),
        "total_strategy_mentions": int(len(all_strategies)),
        "individual_responses": individual_responses  # NEW: Individual response data
    }


def calculate_overall_sct_metrics(df_eval):
    """
    Calculate overall SCT performance metrics across all questions.
    
    Returns:
    - sct_score: Overall SCT score (average of most_frequent_score across all questions)
    - total_questions: Number of questions in dataset
    - self_consistency_overall: Average self-consistency across all questions
    - expert_modal_agreement_rate: Fraction of questions where model agrees with expert consensus
    """
    # Extract case metrics
    case_metrics = [row for row in df_eval['question_stats'] if row is not None]
    
    if not case_metrics:
        return {}
    
    # SCT Score: Average of most frequent scores across all questions
    sct_score = np.mean([metrics['most_frequent_score'] for metrics in case_metrics])
    
    # Total questions
    total_questions = len(case_metrics)
    
    # Overall self-consistency: Average self-consistency across all questions (using most frequent)
    self_consistency_overall = np.mean([metrics['most_frequent_self_consistency'] for metrics in case_metrics])
    
    # Expert modal agreement rate: Fraction of questions where model matches expert consensus
    expert_agreements = [metrics['expert_modal_agreement'] for metrics in case_metrics]
    expert_modal_agreement_rate = np.mean(expert_agreements)
    
    # Additional metrics
    uncertainties = [metrics['most_frequent_uncertainty'] for metrics in case_metrics 
                    if metrics['most_frequent_uncertainty'] is not None]
    avg_most_frequent_uncertainty = np.mean(uncertainties) if uncertainties else None
    
    reasoning_tokens = [metrics['average_reasoning_tokens'] for metrics in case_metrics 
                       if metrics['average_reasoning_tokens'] is not None]
    avg_reasoning_tokens = np.mean(reasoning_tokens) if reasoning_tokens else None
    
    return {
        "sct_score": float(sct_score),
        "total_questions": int(total_questions),
        "self_consistency_overall": float(self_consistency_overall),
        "expert_modal_agreement_rate": float(expert_modal_agreement_rate),
        "avg_most_frequent_uncertainty": float(avg_most_frequent_uncertainty) if avg_most_frequent_uncertainty is not None else None,
        "average_reasoning_tokens": float(avg_reasoning_tokens) if avg_reasoning_tokens is not None else None,
        "questions_with_uncertainty": int(len(uncertainties)),
        "questions_with_reasoning": int(len(reasoning_tokens))
    }

In [ ]:
sct_scores = {}

for model_name, df_eval in processed_data.items():
    print(f"\nCalculating metrics for {model_name}...")
    
    # Calculate case metrics
    df_eval['question_stats'] = df_eval.apply(calculate_sct_metrics, axis=1)
    
    # Save detailed results
    eval_path = os.path.join(EVAL_PATH, f"metrics_{model_name}.jsonl")
    with open(eval_path, "w") as eval_file:
        for _, row in df_eval.iterrows():  # Fixed: removed asterisks
            result = {
                "question_id": row["question_id"],
                "question": row["question"],
                "sct_stem": row["sct_stem"],
                "additional_info": row["additional_info"],
                "expert_distribution": row["expert_distribution"],
                "expert_modal_response": row["expert_modal_response"],
                "response": row["response"],
                "question_stats": row["question_stats"]
            }
            eval_file.write(json.dumps(result) + "\n")
    
    # Calculate overall metrics
    overall_metrics = calculate_overall_sct_metrics(df_eval)
    sct_scores[model_name] = overall_metrics
    
    # Fixed: Updated metric names to match new function
    print(f"  SCT Score: {overall_metrics['sct_score']:.5f}")
    print(f"  Self-Consistency: {overall_metrics['self_consistency_overall']:.3f}")
    print(f"  Expert Agreement: {overall_metrics['expert_modal_agreement_rate']:.3f}")

# Save overall scores
scores_path = os.path.join(EVAL_PATH, "sct_scores.json")
with open(scores_path, "w") as scores_file:
    json.dump(sct_scores, scores_file, indent=2)

print(f"\nMetrics calculated for {len(sct_scores)} models")

In [ ]:
# ============================================================================
# LOAD PROCESSED_DATA FROM EXISTING METRICS FILES
# ============================================================================

def load_sct_metrics_files(eval_path):
    """
    Load SCT metrics files and recreate both processed_data AND sct_scores
    so that all analysis functions work with the same data format.
    """
    processed_data = {}
    sct_scores = {}
    
    print(f"Looking for SCT metrics files in: {eval_path}")
    
    # Find all metrics files
    metrics_files = glob.glob(os.path.join(eval_path, 'metrics_*.jsonl'))
    
    if not metrics_files:
        print(f"No metrics files found in {eval_path}")
        return {}, {}
    
    for filepath in metrics_files:
        filename = os.path.basename(filepath)
        model_name = filename.replace('metrics_', '').replace('.jsonl', '')
        
        print(f"Loading: {filename}")
        
        # Load the metrics data
        metrics_data = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line)
                    metrics_data.append(data)
        
        # Convert to DataFrame format
        df_data = []
        for item in metrics_data:
            # FIX: Convert rating_distribution keys back to integers
            if item['question_stats'] and 'rating_distribution' in item['question_stats']:
                rating_dist = item['question_stats']['rating_distribution']
                # Convert string keys back to integers
                item['question_stats']['rating_distribution'] = {int(k): v for k, v in rating_dist.items()}
            
            # Also fix self_consistency if it has the same issue
            if item['question_stats'] and 'self_consistency' in item['question_stats']:
                self_consistency = item['question_stats']['self_consistency']
                if isinstance(self_consistency, dict):
                    item['question_stats']['self_consistency'] = {int(k): v for k, v in self_consistency.items()}
            
            # Fix rating_distribution_normalized keys too
            if item['question_stats'] and 'rating_distribution_normalized' in item['question_stats']:
                rating_dist_norm = item['question_stats']['rating_distribution_normalized']
                item['question_stats']['rating_distribution_normalized'] = {int(k): v for k, v in rating_dist_norm.items()}
            
            df_data.append({
                'question_id': item['question_id'],
                'question': item['question'],
                'sct_stem': item['sct_stem'],
                'additional_info': item['additional_info'],
                'expert_distribution': item['expert_distribution'],
                'expert_modal_response': item['expert_modal_response'],
                'response': item['response'],
                'results': item.get('results', {}),
                'question_stats': item['question_stats'],
                'source_file': f"{model_name}.json"
            })
        
        df_eval = pd.DataFrame(df_data)
        df_eval['source_file'] = f"{model_name}.json"
        processed_data[model_name] = df_eval
        
        # RECREATE sct_scores for this model
        overall_metrics = calculate_overall_sct_metrics(df_eval)
        sct_scores[model_name] = overall_metrics
        
        print(f"  Loaded {len(df_eval)} questions for {model_name}")
        print(f"  SCT Score: {overall_metrics['sct_score']:.5f}")
    
    print(f"\nLoaded {len(processed_data)} models from metrics files")
    return processed_data, sct_scores

# Use the complete function
processed_data_from_metrics, sct_scores_from_metrics = load_sct_metrics_files(EVAL_PATH)

if processed_data_from_metrics:
    processed_data = processed_data_from_metrics
    sct_scores = sct_scores_from_metrics
    print("Successfully replaced both processed_data and sct_scores with data from metrics files")
else:
    print("Failed to load from metrics files, using original processed_data")

In [ ]:
# ============================================================================
# CREATE SCT VISUALIZATIONS WITH CLEAN LABELS
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
import os

# Prepare data for plotting
model_names = list(sct_scores.keys())
sct_score_values = [sct_scores[name]['sct_score'] for name in model_names]
overall_self_consistency_values = [sct_scores[name]['self_consistency_overall'] for name in model_names]

# Create clean labels for this specific data
plot_labels = get_clean_labels(model_names)

# Filter out None values
valid_sct_scores = [(name, score, label) for name, score, label in zip(model_names, sct_score_values, plot_labels) if score is not None]
valid_self_consistency = [(name, consistency, label) for name, consistency, label in zip(model_names, overall_self_consistency_values, plot_labels) if consistency is not None]

if len(model_names) > 0:
    
    # ============================================================================
    # PLOT 1: SCT SCORES BAR CHART
    # ============================================================================
    
    if valid_sct_scores:
        valid_names_sct = [name for name, score, label in valid_sct_scores]
        valid_scores_sct = [score for name, score, label in valid_sct_scores]
        valid_labels_sct = [label for name, score, label in valid_sct_scores]
        
        plt.figure(figsize=(12, 8))
        
        # Use different colors for each bar
        colors = plt.cm.Set3(np.linspace(0, 1, len(valid_names_sct)))
        bars = plt.bar(range(len(valid_names_sct)), valid_scores_sct, alpha=0.8, 
                       color=colors, edgecolor='black')
        plt.xlabel('Models', fontsize=12)
        plt.ylabel('SCT Score', fontsize=12)
        plt.title('SCT Performance Comparison', fontsize=14, fontweight='bold')
        plt.xticks(range(len(valid_names_sct)), valid_labels_sct, rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for bar, score in zip(bars, valid_scores_sct):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
        
        plt.tight_layout()
        
        # Save the plot
        plots_dir = os.path.join(EVAL_PATH, 'plots')
        os.makedirs(plots_dir, exist_ok=True)
        plot_path_sct = os.path.join(plots_dir, 'sct_scores_comparison.png')
        plt.savefig(plot_path_sct, dpi=300, bbox_inches='tight')
        print(f"SCT scores plot saved to: {plot_path_sct}")
        
        plt.show()
    else:
        print("No SCT score data available for plotting")
    
    # ============================================================================
    # PLOT 2: SELF-CONSISTENCY BOX PLOT
    # ============================================================================
    
    if valid_self_consistency:
        # Create individual self-consistency distributions for box plot
        self_consistency_distributions = []
        valid_names_consistency = []
        valid_labels_consistency = []
        
        for name, consistency, label in valid_self_consistency:
            # Extract individual self-consistency values for this model
            model_df = processed_data[name]
            individual_values = []
            
            for _, row in model_df.iterrows():
                question_stats = row['question_stats']
                if question_stats and 'most_frequent_self_consistency' in question_stats:
                    # Use the self-consistency value for the most frequent rating
                    self_consistency_val = question_stats['most_frequent_self_consistency']
                    if self_consistency_val is not None:
                        individual_values.append(self_consistency_val)
            
            if individual_values:
                self_consistency_distributions.append(individual_values)
                valid_names_consistency.append(name)
                valid_labels_consistency.append(label)
        
        if self_consistency_distributions:
            plt.figure(figsize=(12, 8))
            
            box_plot = plt.boxplot(self_consistency_distributions, tick_labels=valid_labels_consistency, 
                                  patch_artist=True, showmeans=True)
            plt.xlabel('Models', fontsize=12)
            plt.ylabel('Self-Consistency', fontsize=12)
            plt.title('Self-Consistency Distribution', fontsize=14, fontweight='bold')
            plt.xticks(rotation=45, ha='right')
            plt.grid(True, alpha=0.3, axis='y')
            plt.ylim(0, 1)  # Self-consistency is between 0 and 1
            
            # Color the boxplots
            colors = plt.cm.Set3(np.linspace(0, 1, len(valid_names_consistency)))
            for patch, color in zip(box_plot['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            # Add overall mean value labels (from calculate_overall_sct_metrics)
            for i, name in enumerate(valid_names_consistency):
                overall_mean = sct_scores[name]['self_consistency_overall']
                plt.text(i+1, overall_mean, f'{overall_mean:.3f}', ha='center', va='center',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8),
                        fontweight='bold', fontsize=10)
            
            plt.tight_layout()
            
            # Save the plot
            plot_path_consistency = os.path.join(plots_dir, 'sct_self_consistency_distribution.png')
            plt.savefig(plot_path_consistency, dpi=300, bbox_inches='tight')
            print(f"Self-consistency plot saved to: {plot_path_consistency}")
            
            plt.show()
        else:
            print("No individual self-consistency data available for plotting")
    else:
        print("No self-consistency data available for plotting")
    
    # ============================================================================
    # PLOT 3: EXPERT AGREEMENT BAR CHART
    # ============================================================================
    
    expert_agreement_values = [sct_scores[name]['expert_modal_agreement_rate'] for name in model_names]
    valid_expert_agreement = [(name, rate, label) for name, rate, label in zip(model_names, expert_agreement_values, plot_labels) if rate is not None]
    
    if valid_expert_agreement:
        valid_names_expert = [name for name, rate, label in valid_expert_agreement]
        valid_rates_expert = [rate for name, rate, label in valid_expert_agreement]
        valid_labels_expert = [label for name, rate, label in valid_expert_agreement]
        
        plt.figure(figsize=(12, 8))
        
        # Use different colors for each bar
        colors = plt.cm.Set3(np.linspace(0, 1, len(valid_names_expert)))
        bars3 = plt.bar(range(len(valid_names_expert)), valid_rates_expert, alpha=0.8, 
                       color=colors, edgecolor='black')
        plt.xlabel('Models', fontsize=12)
        plt.ylabel('Expert Agreement Rate', fontsize=12)
        plt.title('Expert Modal Agreement Comparison', fontsize=14, fontweight='bold')
        plt.xticks(range(len(valid_names_expert)), valid_labels_expert, rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        plt.ylim(0, 1)  # Agreement rate is between 0 and 1
        
        # Add horizontal line at 50% for reference
        plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, linewidth=2, label='50% baseline')
        
        # Add value labels on bars
        for bar, rate in zip(bars3, valid_rates_expert):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{rate:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
        
        plt.legend(fontsize=11)
        plt.tight_layout()
        
        # Save the plot
        plot_path_expert = os.path.join(plots_dir, 'sct_expert_agreement_comparison.png')
        plt.savefig(plot_path_expert, dpi=300, bbox_inches='tight')
        print(f"Expert agreement plot saved to: {plot_path_expert}")
        
        plt.show()
    else:
        print("No expert agreement data available for plotting")
    
    # Summary
    plots_dir = os.path.join(EVAL_PATH, 'plots')
    os.makedirs(plots_dir, exist_ok=True)
    print(f"\nAll SCT plots saved to: {plots_dir}")
    print(f"  - sct_scores_comparison.png")
    print(f"  - sct_self_consistency_distribution.png") 
    print(f"  - sct_expert_agreement_comparison.png")

In [ ]:
# ===================================================
# CONSENSUS DISTRIBUTION COMPARISON WITH CLEAN LABELS
# ===================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon
from scipy import stats
import os

def normalize_distributions_around_consensus(expert_dist, llm_rating_dist):
    """
    Normalize both expert and LLM distributions around the expert consensus position.
    This preserves the relative alignment/misalignment information.
    
    Args:
        expert_dist: Array of expert weights [weight_-2, weight_-1, weight_0, weight_+1, weight_+2]
        llm_rating_dist: Dict of LLM rating counts {-2: count, -1: count, 0: count, 1: count, 2: count}
    
    Returns:
        Dictionary with normalized distributions centered on expert consensus
    """
    # Find expert consensus (rating with highest weight)
    expert_weights = np.array(expert_dist)
    expert_consensus_idx = np.argmax(expert_weights)
    expert_consensus_rating = expert_consensus_idx - 2  # Convert to -2,+2 scale
    
    # Convert LLM counts to array
    llm_counts = np.array([llm_rating_dist.get(i, 0) for i in range(-2, 3)])
    if llm_counts.sum() == 0:
        return None
    
    # Normalize both distributions to probabilities
    expert_prob = expert_weights / expert_weights.sum()
    llm_prob = llm_counts / llm_counts.sum()
    
    # Create relative position arrays (distance from expert consensus)
    # Expert consensus is always at position 0, other ratings are relative to it
    original_ratings = np.array([-2, -1, 0, 1, 2])
    relative_positions = original_ratings - expert_consensus_rating
    
    # For visualization: extend range to capture potential spread
    extended_positions = np.arange(-4, 5)  # -4 to +4 relative to consensus
    
    # Map probabilities to relative positions
    expert_normalized = np.zeros(len(extended_positions))
    llm_normalized = np.zeros(len(extended_positions))
    
    for i, rel_pos in enumerate(relative_positions):
        # Find index in extended array where rel_pos = 0 is center
        center_idx = len(extended_positions) // 2  # Index of position 0
        target_idx = center_idx + rel_pos
        
        if 0 <= target_idx < len(extended_positions):
            expert_normalized[target_idx] = expert_prob[i]
            llm_normalized[target_idx] = llm_prob[i]
    
    return {
        'expert_consensus_rating': expert_consensus_rating,
        'relative_positions': extended_positions,
        'expert_normalized': expert_normalized,
        'llm_normalized': llm_normalized,
        'expert_center_idx': len(extended_positions) // 2,
        'llm_center': np.average(extended_positions, weights=llm_normalized) if llm_normalized.sum() > 0 else 0
    }


def analyze_alignment_preserving_distributions(processed_data, output_dir=None):
    """
    Analyze distributions using alignment-preserving normalization.
    This captures how well LLM distributions align with expert consensus patterns.
    """
    
    overall_results = {}
    
    # Get model names and create clean labels for this function
    model_names = list(processed_data.keys())
    
    # Use clean_labels if available, otherwise fall back to model names
    try:
        plot_labels = get_clean_labels(model_names)
    except NameError:
        print("Warning: clean_labels not found, using original model names")
        plot_labels = model_names
    
    # Create label mapping
    label_mapping = dict(zip(model_names, plot_labels))
    
    for model_name, df_eval in processed_data.items():
        clean_model_name = label_mapping[model_name]
        print(f"\n=== ALIGNMENT-PRESERVING ANALYSIS: {clean_model_name} ===")
        
        # Collect normalized distributions from all questions
        all_expert_normalized = []
        all_llm_normalized = []
        alignment_metrics = []
        consensus_deviations = []
        
        valid_questions = 0
        
        for _, row in df_eval.iterrows():
            if row['question_stats'] is None:
                continue
                
            expert_dist = row['expert_distribution']
            llm_rating_dist = row['question_stats']['rating_distribution']
            
            normalized = normalize_distributions_around_consensus(expert_dist, llm_rating_dist)
            if normalized is None:
                continue
                
            valid_questions += 1
            
            # Collect normalized distributions
            all_expert_normalized.append(normalized['expert_normalized'])
            all_llm_normalized.append(normalized['llm_normalized'])
            
            # Calculate alignment metrics for this question
            js_distance = jensenshannon(normalized['expert_normalized'], normalized['llm_normalized'])
            
            # How far is LLM center from expert consensus (always at position 0)?
            consensus_deviation = abs(normalized['llm_center'])
            
            alignment_metrics.append(js_distance)
            consensus_deviations.append(consensus_deviation)
        
        if valid_questions == 0:
            print(f"  No valid questions for {clean_model_name}")
            continue
        
        # Create meta-distributions (averages across questions)
        # This preserves alignment information because each question contributes
        # its expert-consensus-centered distribution
        meta_expert_dist = np.mean(all_expert_normalized, axis=0)
        meta_llm_dist = np.mean(all_llm_normalized, axis=0)
        
        # Calculate overall alignment metrics
        meta_js_distance = jensenshannon(meta_expert_dist, meta_llm_dist)
        
        # Expert meta-distribution should peak at center (position 0)
        expert_peak_pos = np.argmax(meta_expert_dist) - len(meta_expert_dist) // 2
        llm_peak_pos = np.argmax(meta_llm_dist) - len(meta_llm_dist) // 2
        
        # Calculate overlap area
        overlap = np.sum(np.minimum(meta_expert_dist, meta_llm_dist))
        
        # Calculate average consensus deviation
        avg_consensus_deviation = np.mean(consensus_deviations)
        
        overall_results[model_name] = {
            'clean_name': clean_model_name,  # Store clean name for plotting
            'valid_questions': valid_questions,
            'meta_expert_dist': meta_expert_dist,
            'meta_llm_dist': meta_llm_dist,
            'meta_js_distance': meta_js_distance,
            'avg_alignment_score': np.mean(alignment_metrics),
            'avg_consensus_deviation': avg_consensus_deviation,
            'expert_peak_position': expert_peak_pos,
            'llm_peak_position': llm_peak_pos,
            'distribution_overlap': overlap,
            'alignment_std': np.std(alignment_metrics),
            'consensus_deviation_std': np.std(consensus_deviations)
        }
        
        print(f"  Valid questions: {valid_questions}")
        print(f"  Average alignment (JS distance): {np.mean(alignment_metrics):.4f}")
        print(f"  Meta-distribution JS distance: {meta_js_distance:.4f}")
        print(f"  Average consensus deviation: {avg_consensus_deviation:.3f}")
        print(f"  Distribution overlap: {overlap:.3f}")
        print(f"  Expert peak at: {expert_peak_pos} (should be 0)")
        print(f"  LLM peak at: {llm_peak_pos} (0=perfect alignment)")
    
    # Create visualizations
    if output_dir and overall_results:
        create_alignment_visualizations_separated(overall_results, output_dir)
    
    return overall_results


def create_alignment_visualizations_separated(overall_results, output_dir):
    """
    Create separate visualizations for alignment-preserving distribution analysis.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    model_names = list(overall_results.keys())
    clean_names = [overall_results[name]['clean_name'] for name in model_names]
    extended_positions = np.arange(-4, 5)  # -4 to +4 relative positions
    
    # ============================================================================
    # PLOT 1: Meta-Distribution Bell Curves - Individual Plots for Each Model
    # ============================================================================
    
    for model_name in model_names:
        plt.figure(figsize=(12, 8))
        
        expert_dist = overall_results[model_name]['meta_expert_dist']
        llm_dist = overall_results[model_name]['meta_llm_dist']
        clean_name = overall_results[model_name]['clean_name']
        
        # Plot as bell curve-like distributions
        plt.plot(extended_positions, expert_dist, 'b-', linewidth=4, alpha=0.8, label='Expert Meta-Distribution')
        plt.fill_between(extended_positions, expert_dist, alpha=0.3, color='blue')
        
        plt.plot(extended_positions, llm_dist, 'r-', linewidth=4, alpha=0.8, label='LLM Meta-Distribution')
        plt.fill_between(extended_positions, llm_dist, alpha=0.3, color='red')
        
        # Mark expert consensus at 0
        plt.axvline(x=0, color='black', linestyle='--', alpha=0.7, linewidth=2, label='Expert Consensus')
        
        plt.title(f'Alignment-Preserving Distribution: {clean_name}\nJS Distance: {overall_results[model_name]["meta_js_distance"]:.3f}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Relative Position from Expert Consensus', fontsize=12)
        plt.ylabel('Probability Density', fontsize=12)
        plt.legend(fontsize=12)
        plt.grid(True, alpha=0.3)
        
        # Set appropriate y-axis limits
        max_y = max(expert_dist.max(), llm_dist.max()) * 1.1
        plt.ylim(0, max_y)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'sct_alignment_distribution_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
    
    # ============================================================================
    # PLOT 2: JS Distance Comparison (Separate Plot)
    # ============================================================================
    
    plt.figure(figsize=(12, 8))
    
    js_distances = [overall_results[name]['meta_js_distance'] for name in model_names]
    colors = plt.cm.Set3(np.linspace(0, 1, len(model_names)))
    
    bars = plt.bar(range(len(model_names)), js_distances, color=colors, alpha=0.8, edgecolor='black')
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Jensen-Shannon Distance', fontsize=12)
    plt.title('Distribution Alignment Comparison\n(Lower = Better)', fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_names, rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    for bar, distance in zip(bars, js_distances):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{distance:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_alignment_js_distance_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # ============================================================================
    # PLOT 3: Consensus Deviation Comparison (Separate Plot)
    # ============================================================================
    
    plt.figure(figsize=(12, 8))
    
    deviations = [overall_results[name]['avg_consensus_deviation'] for name in model_names]
    
    bars = plt.bar(range(len(model_names)), deviations, color=colors, alpha=0.8, edgecolor='black')
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Average Consensus Deviation', fontsize=12)
    plt.title('Distance from Expert Consensus\n(Lower = Better)', fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_names, rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    for bar, dev in zip(bars, deviations):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{dev:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_consensus_deviation_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # ============================================================================
    # PLOT 4: Distribution Overlap Comparison (Separate Plot)
    # ============================================================================
    
    plt.figure(figsize=(12, 8))
    
    overlaps = [overall_results[name]['distribution_overlap'] for name in model_names]
    
    bars = plt.bar(range(len(model_names)), overlaps, color=colors, alpha=0.8, edgecolor='black')
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Distribution Overlap', fontsize=12)
    plt.title('Expert-LLM Distribution Overlap\n(Higher = Better)', fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_names, rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    plt.ylim(0, 1)
    
    for bar, overlap in zip(bars, overlaps):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{overlap:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_distribution_overlap_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nAlignment analysis plots saved to: {output_dir}")
    print(f"  - Individual distribution plots: sct_alignment_distribution_[model].png")
    print(f"  - JS distance comparison: sct_alignment_js_distance_comparison.png")
    print(f"  - Consensus deviation: sct_consensus_deviation_comparison.png")
    print(f"  - Distribution overlap: sct_distribution_overlap_comparison.png")


In [ ]:
# VISUALIZATION CONSENSUS DISTRIBUTION
output_dir = os.path.join(EVAL_PATH, 'plots')
results = analyze_alignment_preserving_distributions(processed_data, output_dir)

In [ ]:
# ============================================================================
# CALIBRATION AND ROC ANALYSIS FUNCTIONS WITH THREE METHODS
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import os
import random
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, roc_auc_score

def flatten_sct_majority(data):
    """SCT majority method - uses most frequent rating"""
    records = []
    for case in data:
        stats = case['question_stats']
        if stats:
            most_frequent_rating = stats['most_frequent_rating']
            most_frequent_uncertainty = stats['most_frequent_uncertainty']
            expert_modal_agreement = stats['expert_modal_agreement']
            
            # Convert uncertainty to confidence (0-1 scale)
            if most_frequent_uncertainty is not None:
                confidence = most_frequent_uncertainty / 100.0
            else:
                confidence = 0.5  # Default neutral confidence
            
            # Get self-consistency for the most frequent rating
            self_consistency = stats['most_frequent_self_consistency']
            
            records.append({
                'question_id': case['question_id'],
                'predicted_rating': most_frequent_rating,
                'confidence': confidence,
                'self_consistency': self_consistency,
                'is_correct': expert_modal_agreement
            })
    return records

def flatten_sct_random(data, random_seed=42):
    """SCT random method - selects one random response per question"""
    random.seed(random_seed)
    records = []
    
    for case in data:
        stats = case['question_stats']
        
        if stats and 'individual_responses' in stats and stats['individual_responses']:
            individual_responses = stats['individual_responses']
            
            # Randomly select one response
            response_keys = list(individual_responses.keys())
            selected_key = random.choice(response_keys)
            selected_response = individual_responses[selected_key]
            
            rating = selected_response['rating']
            uncertainty = selected_response['uncertainty']
            
            # Convert uncertainty to confidence
            if uncertainty is not None:
                confidence = uncertainty / 100.0
            else:
                confidence = 0.5
            
            # Check if this random response matches expert consensus
            expert_consensus = stats['expert_consensus_rating']
            is_correct = (rating == expert_consensus)
            
            # Get self-consistency for this specific rating
            self_consistency = stats['self_consistency'].get(rating, 0)
            
            records.append({
                'question_id': case['question_id'],
                'predicted_rating': rating,
                'confidence': confidence,
                'self_consistency': self_consistency,
                'is_correct': is_correct,
                'selected_response': selected_key
            })
        else:
            # Fallback to majority method
            if stats:
                most_frequent_rating = stats['most_frequent_rating']
                most_frequent_uncertainty = stats['most_frequent_uncertainty']
                expert_modal_agreement = stats['expert_modal_agreement']
                
                confidence = most_frequent_uncertainty / 100.0 if most_frequent_uncertainty is not None else 0.5
                self_consistency = stats['most_frequent_self_consistency']
                
                records.append({
                    'question_id': case['question_id'],
                    'predicted_rating': most_frequent_rating,
                    'confidence': confidence,
                    'self_consistency': self_consistency,
                    'is_correct': expert_modal_agreement,
                    'selected_response': 'majority_fallback'
                })
    
    return records

def flatten_sct_all_responses(data):
    """SCT all responses method - uses every individual response"""
    records = []
    
    for case in data:
        stats = case['question_stats']
        
        if stats and 'individual_responses' in stats and stats['individual_responses']:
            individual_responses = stats['individual_responses']
            expert_consensus = stats['expert_consensus_rating']
            
            # Include ALL responses
            for response_key, response_data in individual_responses.items():
                rating = response_data['rating']
                uncertainty = response_data['uncertainty']
                
                # Convert uncertainty to confidence
                if uncertainty is not None:
                    confidence = uncertainty / 100.0
                else:
                    confidence = 0.5
                
                # Check if this response matches expert consensus
                is_correct = (rating == expert_consensus)
                
                # Get self-consistency for this specific rating
                self_consistency = stats['self_consistency'].get(rating, 0)
                
                records.append({
                    'question_id': case['question_id'],
                    'predicted_rating': rating,
                    'confidence': confidence,
                    'self_consistency': self_consistency,
                    'is_correct': is_correct,
                    'response_key': response_key
                })
        else:
            # Fallback to majority method
            if stats:
                most_frequent_rating = stats['most_frequent_rating']
                most_frequent_uncertainty = stats['most_frequent_uncertainty']
                expert_modal_agreement = stats['expert_modal_agreement']
                
                confidence = most_frequent_uncertainty / 100.0 if most_frequent_uncertainty is not None else 0.5
                self_consistency = stats['most_frequent_self_consistency']
                
                records.append({
                    'question_id': case['question_id'],
                    'predicted_rating': most_frequent_rating,
                    'confidence': confidence,
                    'self_consistency': self_consistency,
                    'is_correct': expert_modal_agreement,
                    'response_key': 'majority_fallback'
                })
    
    return records


def compute_sct_calibration_metrics(y_true, y_conf, n_bins=8):
    """
    Compute ECE, OE, and Brier score using uniform bins.

    Parameters
    ----------
    y_true  : array-like, binary ground-truth (0/1)
    y_conf  : array-like, predicted confidence in [0, 1]
    n_bins  : number of equal-width bins (default 8)

    Returns
    -------
    dict with keys: ECE, OE, Brier
    """
    if len(y_true) != len(y_conf):
        raise ValueError("y_true and y_conf must have same length")

    y_true = np.asarray(y_true)
    y_conf = np.asarray(y_conf)
    y_conf = np.clip(y_conf, 0, 1)

    bins    = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_conf, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)  # handle y_conf == 1.0

    ece, oe, total = 0.0, 0.0, len(y_true)
    for i in range(n_bins):
        mask     = bin_ids == i
        bin_size = int(np.sum(mask))
        if bin_size == 0:
            continue
        avg_conf = float(np.mean(y_conf[mask]))
        avg_acc  = float(np.mean(y_true[mask]))
        ece += (bin_size / total) * np.abs(avg_conf - avg_acc)
        if avg_conf > avg_acc:
            oe += (bin_size / total) * (avg_conf - avg_acc)

    brier = float(np.mean((y_conf - y_true) ** 2))
    return {'ECE': ece, 'OE': oe, 'Brier': brier}

def plot_sct_calibration_and_roc(processed_data, output_dir, signal='confidence',
                                 method='majority', n_bins=15, strategy='uniform'):
    """
    Plot calibration and ROC curves for SCT data.

    Parameters
    ----------
    n_bins    : number of bins for ECE calculation and calibration curve plot
    signal    : 'confidence' or 'self_consistency'
    method    : 'majority', 'random', or 'all_responses'
    strategy  : 'uniform' (equal-width) or 'quantile' (equal-count),
                passed to sklearn calibration_curve for the plot.
                ECE is always computed with uniform bins.
    """

    if method == 'random':
        flatten_func = flatten_sct_random
        method_label = "Random Response"
    elif method == 'all_responses':
        flatten_func = flatten_sct_all_responses
        method_label = "All Individual Responses"
    else:
        flatten_func = flatten_sct_majority
        method_label = "Majority Rating"

    model_names   = list(processed_data.keys())
    plot_labels   = get_clean_labels(model_names)
    label_mapping = dict(zip(model_names, plot_labels))

    os.makedirs(output_dir, exist_ok=True)

    # ============================================================================
    # PLOT 1: Calibration Curve
    # ============================================================================

    plt.figure(figsize=(12, 8))

    calibration_metrics = {}
    for model_name, df_eval in processed_data.items():
        clean_name = label_mapping[model_name]

        data_list      = [row.to_dict() for _, row in df_eval.iterrows()]
        flattened_data = flatten_func(data_list)

        if not flattened_data:
            print(f"Warning: {clean_name} has no data for calibration")
            continue

        y_true = np.array([item['is_correct'] for item in flattened_data])
        y_conf = np.array([item[signal]       for item in flattened_data])

        if len(y_true) == 0:
            print(f"Warning: {clean_name} has no data for calibration")
            continue

        metrics = compute_sct_calibration_metrics(y_true, y_conf, n_bins=n_bins)
        calibration_metrics[model_name] = metrics

        prob_true, prob_pred = calibration_curve(
            y_true, y_conf, n_bins=n_bins, strategy=strategy)

        # hollow circles, size encodes bin occupancy
        bin_sizes = []
        for pp in prob_pred:
            half = (prob_pred[1] - prob_pred[0]) / 2 if len(prob_pred) > 1 else 0.5
            bin_sizes.append(int(np.sum(np.abs(y_conf - pp) <= half)))
        max_bs = max(bin_sizes) if bin_sizes else 1
        sizes = [30 + 270 * (bs / max_bs) for bs in bin_sizes]

        color = plt.gca()._get_lines.get_next_color()
        plt.plot(prob_pred, prob_true, color=color, linewidth=1.5,
                 label=f"{clean_name} (ECE={metrics['ECE']:.3f})")
        plt.scatter(prob_pred, prob_true, s=sizes, facecolors='none',
                    edgecolors=color, linewidths=1.5, zorder=5)

    plt.plot([0, 1], [0, 1], '--', color='gray', alpha=0.8, linewidth=1.5,
             label='Perfect Calibration')
    plt.xlabel("Confidence Score", fontsize=12)
    plt.ylabel("Expert Agreement Rate", fontsize=12)
    strat_label = "quantile bins" if strategy == 'quantile' else "uniform bins"
    #plt.title(f"SCT Calibration Curve - {method_label} (signal: {signal}, {strat_label})",
    #          fontsize=13, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'sct_calibration_curve_{signal}_{method}_bins{n_bins}.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # ============================================================================
    # PLOT 2: ROC Curve
    # ============================================================================

    plt.figure(figsize=(12, 8))

    roc_metrics = {}
    for model_name, df_eval in processed_data.items():
        clean_name = label_mapping[model_name]

        data_list      = [row.to_dict() for _, row in df_eval.iterrows()]
        flattened_data = flatten_func(data_list)

        if not flattened_data:
            continue

        y_true = np.array([item['is_correct'] for item in flattened_data])
        y_conf = np.array([item[signal]       for item in flattened_data])

        if len(y_true) == 0 or len(np.unique(y_true)) <= 1:
            print(f"Warning: {clean_name} has insufficient data for ROC curve")
            continue

        fpr, tpr, _ = roc_curve(y_true, y_conf)
        auc = roc_auc_score(y_true, y_conf)
        roc_metrics[model_name] = auc
        plt.plot(fpr, tpr, linewidth=2, label=f'{clean_name} (AUROC={auc:.3f})')

    plt.plot([0, 1], [0, 1], '--', color='gray', alpha=0.8, linewidth=2, label='Random')
    plt.xlabel("FPR", fontsize=12)
    plt.ylabel("TPR", fontsize=12)
    #plt.title(f"SCT ROC Curve - {method_label} (signal: {signal})",
     #         fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'sct_roc_curve_{signal}_{method}.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n=== SCT CALIBRATION METRICS ({signal.upper()} - {method.upper()}) ===")
    print(f"ECE: uniform bins ({n_bins}), plot: {strat_label} ({n_bins})")
    for model_name in calibration_metrics:
        clean_name  = label_mapping[model_name]
        cal         = calibration_metrics[model_name]
        roc_val     = roc_metrics.get(model_name, "N/A")
        print(f"{clean_name}:")
        print(f"  ECE:   {cal['ECE']:.4f}")
        print(f"  OE:    {cal['OE']:.4f}")
        print(f"  Brier: {cal['Brier']:.4f}")
        print(f"  AUROC: {roc_val:.4f}" if roc_val != "N/A" else "  AUROC: N/A")
        print()

    print(f"Plots saved to: {output_dir}")
    print(f"  - sct_calibration_curve_{signal}_{method}_bins{n_bins}.png")
    print(f"  - sct_roc_curve_{signal}_{method}.png")

    return calibration_metrics, roc_metrics

In [ ]:
output_dir = os.path.join(EVAL_PATH, 'plots')

plot_sct_calibration_and_roc(processed_data, output_dir, signal='self_consistency', method='all_responses', n_bins=8, strategy='uniform')
plot_sct_calibration_and_roc(processed_data, output_dir, signal='self_consistency', method='majority',      n_bins=8, strategy='uniform')
plot_sct_calibration_and_roc(processed_data, output_dir, signal='confidence', method='all_responses', n_bins=8, strategy='uniform')
plot_sct_calibration_and_roc(processed_data, output_dir, signal='confidence', method='majority',      n_bins=8, strategy='uniform')

In [ ]:
def sct_score_analysis_from_metrics(processed_data, N=100, output_dir=None):
    """
    Complete SCT analysis using metrics files, comparing:
    1. Overall SCT scores (calculated using most frequent rating - majority vote)
    2. Random sampling SCT scores (using individual responses from metrics)
    
    Args:
        processed_data: Dictionary of model_name -> DataFrame from metrics files
        N: Number of random samples for individual response analysis
        output_dir: Output directory for plots (optional)
    """
    
    # ============================================================================
    # 1. SETUP OUTPUT DIRECTORY
    # ============================================================================
    
    if output_dir is None:
        output_dir = os.path.join(EVAL_PATH, 'plots')
    os.makedirs(output_dir, exist_ok=True)
    print(f"Plots will be saved to: {output_dir}")
    
    # Get model names and create clean labels
    model_names = list(processed_data.keys())
    
    # Use your existing clean label function
    clean_labels = get_clean_labels(model_names)
    label_mapping = dict(zip(model_names, clean_labels))
    
    all_results = {}
    
    # ============================================================================
    # 2. PROCESS EACH MODEL USING METRICS DATA
    # ============================================================================
    
    for model_name, df_eval in processed_data.items():
        clean_name = label_mapping[model_name]
        print(f"\nProcessing: {clean_name}")
        
        # ============================================================================
        # 3. CALCULATE MAJORITY VOTE SCT SCORE (FROM METRICS)
        # ============================================================================
        
        majority_scores = []
        questions_with_individual_responses = []
        
        for _, row in df_eval.iterrows():
            question_stats = row['question_stats']
            if question_stats:
                # Get majority vote score directly from metrics
                most_frequent_score = question_stats['most_frequent_score']
                majority_scores.append(most_frequent_score)
                
                # Store questions that have individual responses for random sampling
                if 'individual_responses' in question_stats and question_stats['individual_responses']:
                    questions_with_individual_responses.append({
                        'question_id': row['question_id'],
                        'individual_responses': question_stats['individual_responses'],
                        'expert_distribution': row['expert_distribution']
                    })
        
        if not majority_scores:
            print(f"  No valid questions found for {clean_name}")
            continue
        
        # Calculate majority vote SCT score
        overall_sct_score = np.mean(majority_scores)
        print(f"  Majority vote SCT score: {overall_sct_score:.4f} (average of {len(majority_scores)} questions)")
        
        # ============================================================================
        # 4. CALCULATE RANDOM SAMPLING SCT SCORES USING INDIVIDUAL RESPONSES
        # ============================================================================
        
        if not questions_with_individual_responses:
            print(f"  No individual responses available for {clean_name}")
            all_results[model_name] = {
                'clean_name': clean_name,
                'majority_sct_score': overall_sct_score,
                'random_sct_scores': [],
                'random_mean': overall_sct_score,  # Fallback to majority score
                'random_std': 0.0,
                'n_questions': len(majority_scores),
                'n_samples': 0
            }
            continue
        
        print(f"  Calculating {N} random samples from {len(questions_with_individual_responses)} questions with individual responses...")
        random_sct_scores = []
        
        for sample_num in range(N):
            sample_scores = []
            
            for question in questions_with_individual_responses:
                individual_responses = question['individual_responses']
                expert_dist = np.array(question['expert_distribution'])
                
                if individual_responses:
                    # Randomly pick one response
                    random_response_key = random.choice(list(individual_responses.keys()))
                    random_response = individual_responses[random_response_key]
                    random_rating = random_response['rating']
                    
                    # Calculate score for this rating using expert distribution
                    # (same logic as in process_sct_response)
                    expert_dist_normalized = expert_dist / expert_dist.max() if expert_dist.max() > 0 else expert_dist
                    rating_idx = random_rating + 2  # Convert -2,+2 range to 0,4 index
                    score = expert_dist_normalized[rating_idx]
                    
                    sample_scores.append(score)
            
            if sample_scores:
                # Calculate mean SCT score for this random sample
                sample_sct_score = np.mean(sample_scores)
                random_sct_scores.append(sample_sct_score)
        
        random_mean = np.mean(random_sct_scores) if random_sct_scores else overall_sct_score
        random_std = np.std(random_sct_scores) if random_sct_scores else 0.0
        
        print(f"  Random sampling SCT score: {random_mean:.4f} ± {random_std:.4f} (mean ± std of {len(random_sct_scores)} samples)")
        
        # Store results
        all_results[model_name] = {
            'clean_name': clean_name,
            'majority_sct_score': overall_sct_score,
            'random_sct_scores': random_sct_scores,
            'random_mean': random_mean,
            'random_std': random_std,
            'n_questions': len(majority_scores),
            'n_samples': len(random_sct_scores)
        }
    
    # ============================================================================
    # 5. CREATE VISUALIZATIONS - SEPARATE PLOTS WITH CLEAN LABELS
    # ============================================================================
    
    if not all_results:
        print("No results to plot")
        return
    
    # Prepare data for plotting
    file_names = list(all_results.keys())
    clean_names = [all_results[name]['clean_name'] for name in file_names]
    majority_scores = [all_results[name]['majority_sct_score'] for name in file_names]
    
    # ---- Plot 1: Bar chart for majority vote SCT scores (separate plot) ----
    plt.figure(figsize=(12, 8))
    
    # Use different colors for each bar
    colors = plt.cm.Set3(np.linspace(0, 1, len(file_names)))
    bars = plt.bar(range(len(file_names)), majority_scores, alpha=0.8, 
                   color=colors, edgecolor='black')
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('SCT Score', fontsize=12)
    plt.title('Majority Vote SCT Scores', fontsize=14, fontweight='bold')
    plt.xticks(range(len(file_names)), clean_names, rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, score in zip(bars, majority_scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_majority_vote_scores.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # ---- Plot 2: Boxplot for random sampling SCT scores (separate plot) ----
    # Only plot if we have random sampling data
    models_with_random_data = [name for name in file_names if all_results[name]['n_samples'] > 0]
    
    if models_with_random_data:
        plt.figure(figsize=(12, 8))
        
        random_data = [all_results[name]['random_sct_scores'] for name in models_with_random_data]
        random_clean_names = [all_results[name]['clean_name'] for name in models_with_random_data]
        
        box_plot = plt.boxplot(random_data, tick_labels=random_clean_names, patch_artist=True)
        plt.xlabel('Models', fontsize=12)
        plt.ylabel('SCT Score', fontsize=12)
        plt.title(f'Random Sampling SCT Score Distribution\n({N} samples per model)', 
                  fontsize=14, fontweight='bold')
        plt.xticks(rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        
        # Color the boxplots
        colors = plt.cm.Set3(np.linspace(0, 1, len(models_with_random_data)))
        for patch, color in zip(box_plot['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Add mean value labels
        for i, name in enumerate(models_with_random_data):
            mean_val = all_results[name]['random_mean']
            plt.text(i+1, mean_val, f'{mean_val:.3f}', ha='center', va='center',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8),
                    fontweight='bold', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'sct_random_sampling_distribution_N{N}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print("No models have individual response data for random sampling plot")
    
    # ============================================================================
    # SUMMARY TABLE
    # ============================================================================
    
    print("\n" + "="*90)
    print("SUMMARY RESULTS")
    print("="*90)
    print(f"{'Model Name':<35} {'Majority SCT':<12} {'Random Mean':<12} {'Random Std':<12} {'Questions':<10}")
    print("-" * 90)
    
    for name in file_names:
        result = all_results[name]
        print(f"{result['clean_name']:<35} {result['majority_sct_score']:9.4f} {result['random_mean']:9.4f} "
              f"{result['random_std']:9.4f} {result['n_questions']:8d}")
    
    print("="*90)
    print(f"\n📁 Plots saved to: {output_dir}")
    print(f"  - sct_majority_vote_scores.png")
    if models_with_random_data:
        print(f"  - sct_random_sampling_distribution_N{N}.png")
    
    return all_results

In [ ]:
# Call sct score analysis
sct_results = sct_score_analysis_from_metrics(processed_data, N=100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import re
from collections import Counter

def normalize_strategy_to_official(strategy):
    """
    Normalize strategy names to map to the 12 official clinical reasoning categories.
    Returns 'other' for strategies that don't clearly fit the official categories.
    """
    if not strategy or not isinstance(strategy, str):
        return "other"
    
    # Convert to lowercase and clean
    normalized = strategy.lower().strip()
    normalized = re.sub(r'[^\w\s-]', '', normalized)  # Remove punctuation except hyphens
    normalized = re.sub(r'\s+', ' ', normalized)  # Normalize whitespace
    normalized = normalized.strip()
    
    # Official strategy categories (12 total: 10 + 2 new)
    official_categories = {
        'deductive': 'Deductive Reasoning',
        'hypothetico-deductive': 'Hypothetico-Deductive Reasoning', 
        'inductive': 'Inductive Reasoning',
        'abductive': 'Abductive Reasoning',
        'probabilistic': 'Probabilistic Reasoning',
        'rule-based': 'Rule-Based / Categorical / Deterministic Reasoning',
        'causal': 'Causal Reasoning',
        'heuristic': 'Heuristic / Pattern Recognition (Fast Thinking)',
        'red-flag': 'Red Flag / Rule-Out Reasoning',
        'guideline-based': 'Guideline-Based Reasoning',
        'ethical': 'Ethical Reasoning',
        'risk-benefit': 'Risk-Benefit Analysis'
    }
    
    # Comprehensive mapping to official categories
    strategy_mapping = {
        # Deductive Reasoning
        'deductive': 'deductive',
        'deduction': 'deductive',
        'deductive reasoning': 'deductive',
        'top down': 'deductive',
        'top-down': 'deductive',
        'rule application': 'deductive',
        'principle based': 'deductive',
        'principle-based': 'deductive',
        
        # Hypothetico-Deductive Reasoning
        'hypothetico-deductive': 'hypothetico-deductive',
        'hypothetico deductive': 'hypothetico-deductive',
        'hypothetical-deductive': 'hypothetico-deductive',
        'hypothetical deductive': 'hypothetico-deductive',
        'hypothesis testing': 'hypothetico-deductive',
        'hypothesis-driven': 'hypothetico-deductive',
        'hypothesis driven': 'hypothetico-deductive',
        'differential reasoning': 'hypothetico-deductive',
        'multiple hypothesis': 'hypothetico-deductive',
        'hypothetico-deductive reasoning': 'hypothetico-deductive',
        'differential diagnosis': 'hypothetico-deductive',
        'differential': 'hypothetico-deductive',
        
        # Inductive Reasoning
        'inductive': 'inductive',
        'induction': 'inductive',
        'inductive reasoning': 'inductive',
        'bottom up': 'inductive',
        'bottom-up': 'inductive',
        'data driven': 'inductive',
        'data-driven': 'inductive',
        'pattern synthesis': 'inductive',
        
        # Abductive Reasoning
        'abductive': 'abductive',
        'abduction': 'abductive',
        'abductive reasoning': 'abductive',
        'inference to best explanation': 'abductive',
        'best explanation': 'abductive',
        'backwards reasoning': 'abductive',
        'reverse reasoning': 'abductive',
        'explanatory reasoning': 'abductive',
        
        # Probabilistic Reasoning
        'probabilistic': 'probabilistic',
        'probability': 'probabilistic',
        'probabilistic reasoning': 'probabilistic',
        'bayesian': 'probabilistic',
        'likelihood': 'probabilistic',
        'statistical': 'probabilistic',
        'risk based': 'probabilistic',
        'risk-based': 'probabilistic',
        'pretest probability': 'probabilistic',
        'post-test probability': 'probabilistic',
        'prevalence': 'probabilistic',
        
        # Rule-Based / Categorical / Deterministic Reasoning
        'rule-based': 'rule-based',
        'rule based': 'rule-based',
        'categorical': 'rule-based',
        'deterministic': 'rule-based',
        'criteria based': 'rule-based',
        'criteria-based': 'rule-based',
        'threshold': 'rule-based',
        'cutoff': 'rule-based',
        'scoring system': 'rule-based',
        'decision rule': 'rule-based',
        'clinical decision rule': 'rule-based',
        'algorithmic': 'rule-based',
        'algorithm': 'rule-based',
        
        # Causal Reasoning (expanded to include anatomical/physiological knowledge)
        'causal': 'causal',
        'causation': 'causal',
        'causal reasoning': 'causal',
        'pathophysiologic': 'causal',
        'pathophysiology': 'causal',
        'pathophysiological': 'causal',
        'mechanistic': 'causal',
        'mechanism': 'causal',
        'cause effect': 'causal',
        'cause-effect': 'causal',
        'physiologic': 'causal',
        'physiological': 'causal',
        'anatomical': 'causal',
        'anatomy': 'causal',
        'anatomical knowledge': 'causal',
        'physiological knowledge': 'causal',
        'pathophysiological knowledge': 'causal',
        'biochemical': 'causal',
        'molecular': 'causal',
        'pharmacological': 'causal',
        'pharmacology': 'causal',
        'pharmacokinetic': 'causal',
        'pharmacodynamic': 'causal',
        
        # Heuristic / Pattern Recognition (Fast Thinking)
        'heuristic': 'heuristic',
        'heuristics': 'heuristic',
        'pattern recognition': 'heuristic',
        'pattern matching': 'heuristic',
        'pattern': 'heuristic',
        'recognition': 'heuristic',
        'intuitive': 'heuristic',
        'intuition': 'heuristic',
        'fast thinking': 'heuristic',
        'system 1': 'heuristic',
        'gut feeling': 'heuristic',
        'clinical intuition': 'heuristic',
        'gestalt': 'heuristic',
        
        # Red Flag / Rule-Out Reasoning
        'red flag': 'red-flag',
        'red-flag': 'red-flag',
        'rule out': 'red-flag',
        'rule-out': 'red-flag',
        'ruling out': 'red-flag',
        'exclusion': 'red-flag',
        'exclude': 'red-flag',
        'worst case': 'red-flag',
        'worst-case': 'red-flag',
        'emergency': 'red-flag',
        'critical': 'red-flag',
        'life threatening': 'red-flag',
        'life-threatening': 'red-flag',
        'safety first': 'red-flag',
        'dont miss': 'red-flag',
        'cannot miss': 'red-flag',
        
        # Guideline-Based Reasoning
        'guideline-based': 'guideline-based',
        'guideline based': 'guideline-based',
        'guideline': 'guideline-based',
        'guidelines': 'guideline-based',
        'protocol': 'guideline-based',
        'protocols': 'guideline-based',
        'evidence-based': 'guideline-based',
        'evidence based': 'guideline-based',
        'standard of care': 'guideline-based',
        'best practice': 'guideline-based',
        'best practices': 'guideline-based',
        'recommendation': 'guideline-based',
        'recommendations': 'guideline-based',
        'consensus': 'guideline-based',
        
        # NEW: Ethical Reasoning
        'ethical': 'ethical',
        'ethics': 'ethical',
        'ethical reasoning': 'ethical',
        'medical ethics': 'ethical',
        'bioethics': 'ethical',
        'ethical principles': 'ethical',
        'autonomy': 'ethical',
        'beneficence': 'ethical',
        'non-maleficence': 'ethical',
        'nonmaleficence': 'ethical',
        'justice': 'ethical',
        'veracity': 'ethical',
        'informed consent': 'ethical',
        'patient rights': 'ethical',
        'professional integrity': 'ethical',
        'moral': 'ethical',
        'morality': 'ethical',
        
        # NEW: Risk-Benefit Analysis
        'risk': 'risk-benefit',
        'benefit': 'risk-benefit',
        'risk benefit': 'risk-benefit',
        'risk-benefit': 'risk-benefit',
        'patient centered': 'risk-benefit',
        'patient-centered': 'risk-benefit',
        'patient benefit': 'risk-benefit',
        'patient risk': 'risk-benefit',
        'harm': 'risk-benefit',
        'adverse effects': 'risk-benefit',
        'side effects': 'risk-benefit',
        'complications': 'risk-benefit',
        'patient safety': 'risk-benefit',
        'quality of life': 'risk-benefit',
        'patient preferences': 'risk-benefit',
        'shared decision': 'risk-benefit',
        'individualized': 'risk-benefit',
        'personalized': 'risk-benefit',
    }
    
    # Try exact match first
    if normalized in strategy_mapping:
        category = strategy_mapping[normalized]
        return official_categories[category]
    
    # Try partial matches for compound strategies - LONGEST FIRST to avoid conflicts
    # Sort by length descending so "hypothetico-deductive" is checked before "deductive"
    sorted_keys = sorted(strategy_mapping.items(), key=lambda x: len(x[0]), reverse=True)
    for key, category in sorted_keys:
        if key in normalized:
            return official_categories[category]
    
    # Check if it contains key words from official categories (also longest first)
    for key, category in sorted_keys:
        if any(word in normalized for word in key.split()):
            return official_categories[category]
    
    # If no mapping found, return 'other'
    return "other"


def analyze_sct_strategies_official_categories(processed_data, output_dir):
    """
    Analyze SCT reasoning strategies using the official 12 categories plus 'other'.
    Updated to include Ethical Reasoning and Risk-Benefit Analysis, with clean labels.
    """
    
    # Get model names and create clean labels for this function
    model_names = list(processed_data.keys())
    
    # Use clean_labels if available, otherwise fall back to model names
    try:
        plot_labels = clean_labels[:len(model_names)] if len(clean_labels) >= len(model_names) else model_names
    except NameError:
        print("Warning: clean_labels not found, using original model names")
        plot_labels = model_names
    
    # Create label mapping
    label_mapping = dict(zip(model_names, plot_labels))
    
    for model_name, df_eval in processed_data.items():
        clean_model_name = label_mapping[model_name]
        print(f"\n=== SCT OFFICIAL CATEGORY ANALYSIS: {clean_model_name} ===")
        
        # Collect and categorize all strategy data
        all_strategies = []
        strategies_agree = []
        strategies_disagree = []
        
        # Track original to official category mapping
        category_variations = {}
        unmapped_strategies = set()
        
        for _, row in df_eval.iterrows():
            if row['question_stats'] and 'strategies' in row['question_stats']:
                strategies = row['question_stats']['strategies']
                # For SCT: use expert_modal_agreement instead of agrees_with_expert_modal
                agrees_with_expert = row['question_stats']['expert_modal_agreement']
                
                # Categorize strategies for this question
                question_strategies = []
                for original_strategy, count in strategies.items():
                    official_category = normalize_strategy_to_official(original_strategy)
                    
                    # Track variations for debugging
                    if official_category not in category_variations:
                        category_variations[official_category] = set()
                    category_variations[official_category].add(original_strategy)
                    
                    # Track unmapped strategies
                    if official_category == "other":
                        unmapped_strategies.add(original_strategy)
                    
                    question_strategies.extend([official_category] * count)
                
                all_strategies.extend(question_strategies)
                
                # Separate by agreement with expert modal response
                if agrees_with_expert:
                    strategies_agree.extend(question_strategies)
                else:
                    strategies_disagree.extend(question_strategies)
        
        if not all_strategies:
            print(f"No strategy data found for {clean_model_name}")
            continue
            
        print(f"Total strategy mentions: {len(all_strategies)}")
        print(f"Strategies when agreeing with expert modal: {len(strategies_agree)}")
        print(f"Strategies when disagreeing with expert modal: {len(strategies_disagree)}")
        
        # Print mapping results
        print(f"\n--- Official Category Mapping Results ---")
        for category, variations in sorted(category_variations.items()):
            if category != "other" and len(variations) > 0:
                print(f"'{category}': {len(variations)} variations")
                if len(variations) <= 5:  # Show all if few
                    print(f"  → {sorted(variations)}")
                else:  # Show sample if many
                    sample = sorted(list(variations))[:3]
                    print(f"  → {sample} ... (+{len(variations)-3} more)")
        
        if unmapped_strategies:
            print(f"\nUnmapped strategies (categorized as 'other'): {len(unmapped_strategies)}")
            for strategy in sorted(unmapped_strategies):
                print(f"  → '{strategy}'")
        
        # Count strategies by official categories
        strategy_counts_all = Counter(all_strategies)
        strategy_counts_agree = Counter(strategies_agree)
        strategy_counts_disagree = Counter(strategies_disagree)
        
        # Get official categories in a consistent order (updated to 12 categories)
        official_order = [
            'Deductive Reasoning',
            'Hypothetico-Deductive Reasoning',
            'Inductive Reasoning', 
            'Abductive Reasoning',
            'Probabilistic Reasoning',
            'Rule-Based / Categorical / Deterministic Reasoning',
            'Causal Reasoning',
            'Heuristic / Pattern Recognition (Fast Thinking)',
            'Red Flag / Rule-Out Reasoning',
            'Guideline-Based Reasoning',
            'Ethical Reasoning',
            'Risk-Benefit Analysis',
            'other'
        ]
        
        # Filter to only categories that were actually used
        used_categories = [cat for cat in official_order if strategy_counts_all[cat] > 0]
        
        # Create plots with shorter labels for readability (updated to remove "Reasoning")
        short_labels = {
            'Deductive Reasoning': 'Deductive',
            'Hypothetico-Deductive Reasoning': 'Hypothetico-\nDeductive',
            'Inductive Reasoning': 'Inductive',
            'Abductive Reasoning': 'Abductive', 
            'Probabilistic Reasoning': 'Probabilistic',
            'Rule-Based / Categorical / Deterministic Reasoning': 'Rule-Based /\nCategorical',
            'Causal Reasoning': 'Causal',
            'Heuristic / Pattern Recognition (Fast Thinking)': 'Heuristic /\nPattern Recog',
            'Red Flag / Rule-Out Reasoning': 'Red Flag /\nRule-Out',
            'Guideline-Based Reasoning': 'Guideline-\nBased',
            'Ethical Reasoning': 'Ethical',
            'Risk-Benefit Analysis': 'Risk-Benefit\nAnalysis',
            'other': 'Other'
        }
        
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # ============================================================================
        # PLOT 1: Overall Distribution (Separate Plot)
        # ============================================================================
        
        plt.figure(figsize=(12, 8))
        
        counts = [strategy_counts_all[cat] for cat in used_categories]
        labels = [short_labels[cat] for cat in used_categories]
        
        colors = plt.cm.Set3(np.arange(len(used_categories)) % 12)
        bars1 = plt.bar(range(len(used_categories)), counts, alpha=0.8, color=colors)
        plt.title(f'Distribution of Reasoning Strategies - GPT-4.1 CoT', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Strategy Categories', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.xticks(range(len(used_categories)), labels, rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        
        # Add count labels
        for bar, count in zip(bars1, counts):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                    str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'sct_strategy_distribution_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # ============================================================================
        # PLOT 2: Agreement vs Disagreement (Separate Plot)
        # ============================================================================
        
        plt.figure(figsize=(12, 8))
        
        agree_counts = [strategy_counts_agree.get(cat, 0) for cat in used_categories]
        disagree_counts = [strategy_counts_disagree.get(cat, 0) for cat in used_categories]
        
        x = np.arange(len(used_categories))
        width = 0.35
        
        bars2a = plt.bar(x - width/2, agree_counts, width, label='Agrees with Expert Modal', 
                        alpha=0.8, color='lightgreen')
        bars2b = plt.bar(x + width/2, disagree_counts, width, label='Disagrees with Expert Modal', 
                        alpha=0.8, color='lightcoral')
        
        plt.title(f'Strategy Usage by Expert Modal Agreement - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Strategy Categories', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.xticks(x, labels, rotation=45, ha='right')
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'sct_strategy_agreement_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # ============================================================================
        # PLOT 3: Effectiveness Rates (Separate Plot)
        # ============================================================================
        
        plt.figure(figsize=(12, 8))
        
        success_rates = []
        for cat in used_categories:
            agree_count = strategy_counts_agree.get(cat, 0)
            total_count = strategy_counts_all.get(cat, 0)
            rate = agree_count / total_count * 100 if total_count > 0 else 0
            success_rates.append(rate)
        
        colors = ['darkgreen' if rate >= 60 else 'gold' if rate >= 40 else 'lightcoral' 
                 for rate in success_rates]
        
        bars3 = plt.bar(range(len(used_categories)), success_rates, alpha=0.8, color=colors)
        plt.title(f'Strategy Effectiveness (% Expert Modal Agreement) - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Strategy Categories', fontsize=12)
        plt.ylabel('Expert Modal Agreement Rate (%)', fontsize=12)
        plt.xticks(range(len(used_categories)), labels, rotation=45, ha='right')
        plt.axhline(y=50, color='red', linestyle='--', alpha=0.7, linewidth=2, label='50% baseline')
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3, axis='y')
        
        for bar, rate in zip(bars3, success_rates):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{rate:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'sct_strategy_effectiveness_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # ============================================================================
        # PLOT 4: Pie Chart (Separate Plot)
        # ============================================================================
        
        plt.figure(figsize=(10, 10))
        
        wedges, texts, autotexts = plt.pie(counts, labels=labels, autopct='%1.1f%%', 
                                          colors=plt.cm.Set3(np.linspace(0, 1, len(used_categories))))
        plt.title(f'Strategy Distribution (Proportional) - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'sct_strategy_pie_chart_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print summary
        print(f"\n--- Official Category Summary for {clean_model_name} ---")
        print(f"Strategies mapped to official categories: {len([s for s in all_strategies if s != 'other'])}")
        print(f"Strategies categorized as 'other': {strategy_counts_all['other']}")
        
        print(f"\nOfficial category usage (ranked by frequency):")
        for i, (category, count) in enumerate(strategy_counts_all.most_common(), 1):
            if count > 0:
                success_rate = strategy_counts_agree.get(category, 0) / count * 100
                print(f"  {i:2d}. {category}: {count} uses, {success_rate:.1f}% success")
        
        print(f"\nStrategy analysis plots saved to: {output_dir}")
        print(f"  - sct_strategy_distribution_{model_name}.png")
        print(f"  - sct_strategy_agreement_{model_name}.png")
        print(f"  - sct_strategy_effectiveness_{model_name}.png")
        print(f"  - sct_strategy_pie_chart_{model_name}.png")

In [ ]:
output_dir = os.path.join(EVAL_PATH, 'plots')

analyze_sct_strategies_official_categories(processed_data, output_dir)

In [ ]:
# ============================================================================
# SCT RESPONSE VARIANCE ANALYSIS  
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import os

def calculate_sct_variance(processed_data, output_dir):
    """
    Calculate response variance for SCT data using rating values (-2 to +2).
    """
    results = {}
    
    for model_name, df_eval in processed_data.items():
        question_variances = []
        question_entropies = []
        
        for _, row in df_eval.iterrows():
            question_stats = row['question_stats']
            
            if question_stats and 'individual_responses' in question_stats:
                individual_responses = question_stats['individual_responses']
                
                if individual_responses and len(individual_responses) > 1:
                    # Get rating values
                    ratings = []
                    for response_data in individual_responses.values():
                        rating = response_data['rating']
                        ratings.append(rating)
                    
                    if len(ratings) > 1:
                        # Calculate variance
                        variance = np.var(ratings, ddof=1)  # Sample variance
                        question_variances.append(variance)
                        
                        # Calculate entropy
                        rating_counts = Counter(ratings)
                        total = len(ratings)
                        entropy = 0
                        for count in rating_counts.values():
                            if count > 0:
                                prob = count / total
                                entropy -= prob * np.log2(prob)
                        question_entropies.append(entropy)
        
        if question_variances:
            results[model_name] = {
                'mean_variance': np.mean(question_variances),
                'std_variance': np.std(question_variances),
                'mean_entropy': np.mean(question_entropies),
                'std_entropy': np.std(question_entropies),
                'question_variances': question_variances,
                'question_entropies': question_entropies,
                'n_questions': len(question_variances)
            }
    
    # Create visualizations
    if results:
        plot_sct_variance(results, output_dir)
    
    return results

def plot_sct_variance(results, output_dir):
    """Create variance and entropy plots for SCT"""
    os.makedirs(output_dir, exist_ok=True)
    
    model_names = list(results.keys())
    clean_labels = get_clean_labels(model_names)
    
    # Plot 1: Mean Variance Comparison
    plt.figure(figsize=(10, 6))
    
    variances = [results[model]['mean_variance'] for model in model_names]
    std_errors = [results[model]['std_variance'] / np.sqrt(results[model]['n_questions']) 
                  for model in model_names]
    
    bars = plt.bar(range(len(model_names)), variances, yerr=std_errors, 
                   capsize=5, alpha=0.8, color=['lightgreen', 'gold'][:len(model_names)])
    
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Mean Response Variance', fontsize=12)
    plt.title('SCT: Response Variance Comparison\n(Rating Scale: -2 to +2)', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_labels)
    plt.grid(True, alpha=0.3, axis='y')
    
    for bar, var in zip(bars, variances):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{var:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_variance_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot 2: Mean Entropy Comparison
    plt.figure(figsize=(10, 6))
    
    entropies = [results[model]['mean_entropy'] for model in model_names]
    entropy_std_errors = [results[model]['std_entropy'] / np.sqrt(results[model]['n_questions']) 
                          for model in model_names]
    
    bars = plt.bar(range(len(model_names)), entropies, yerr=entropy_std_errors, 
                   capsize=5, alpha=0.8, color=['mediumseagreen', 'orange'][:len(model_names)])
    
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Mean Response Entropy', fontsize=12)
    plt.title('SCT: Response Entropy', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_labels)
    plt.grid(True, alpha=0.3, axis='y')
    
    for bar, ent in zip(bars, entropies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{ent:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_entropy_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot 3: Variance Distribution
    plt.figure(figsize=(10, 6))
    
    variance_data = [results[model]['question_variances'] for model in model_names]
    
    box_plot = plt.boxplot(variance_data, tick_labels=clean_labels, patch_artist=True)
    plt.ylabel('Response Variance per Question', fontsize=12)
    plt.title('SCT: Distribution of Response Variance Across Questions', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    colors = ['lightgreen', 'gold']
    for patch, color in zip(box_plot['boxes'], colors[:len(model_names)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_variance_distribution.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot 4: Entropy Distribution
    plt.figure(figsize=(10, 6))
    
    entropy_data = [results[model]['question_entropies'] for model in model_names]
    
    box_plot = plt.boxplot(entropy_data, tick_labels=clean_labels, patch_artist=True)
    plt.ylabel('Response Entropy per Question', fontsize=12)
    plt.title('SCT: Distribution of Response Entropy Across Questions', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    colors = ['mediumseagreen', 'orange']
    for patch, color in zip(box_plot['boxes'], colors[:len(model_names)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_entropy_distribution.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print results
    print("\n" + "="*60)
    print("SCT VARIANCE ANALYSIS RESULTS")
    print("="*60)
    for model_name in model_names:
        clean_name = get_clean_labels([model_name])[0]
        r = results[model_name]
        print(f"{clean_name}:")
        print(f"  Mean Variance: {r['mean_variance']:.4f} (±{r['std_variance']:.4f})")
        print(f"  Mean Entropy: {r['mean_entropy']:.4f} (±{r['std_entropy']:.4f})")
        print(f"  Questions: {r['n_questions']}")
        print()



# ============================================================================
# SIMPLE ADDITION TO YOUR EXISTING SCRIPT - NO FILE LOADING NEEDED
# ============================================================================

def calculate_expert_entropy(expert_distribution):
    """
    Calculate entropy for expert distribution.
    Handles both normalized and raw count distributions.
    """
    dist = np.array(expert_distribution)
    
    # Skip if no expert responses
    if dist.sum() == 0:
        return 0
    
    # Normalize by SUM to get proper probabilities (not by max like SCT scoring)
    probabilities = dist / dist.sum()
    
    # Calculate entropy
    entropy = 0
    for prob in probabilities:
        if prob > 0:
            entropy -= prob * np.log2(prob)
    
    return entropy

def calculate_sct_variance_with_experts(processed_data, output_dir):
    """
    Enhanced version that adds expert entropy to your existing analysis.
    Uses data already in processed_data - no file loading needed.
    """
    results = {}
    all_expert_entropies = []  # Collect all expert entropies across all questions
    
    for model_name, df_eval in processed_data.items():
        question_variances = []
        question_entropies = []
        expert_entropies_for_model = []
        
        for _, row in df_eval.iterrows():
            question_stats = row['question_stats']
            
            # Your existing model response analysis (unchanged)
            if question_stats and 'individual_responses' in question_stats:
                individual_responses = question_stats['individual_responses']
                
                if individual_responses and len(individual_responses) > 1:
                    # Get rating values
                    ratings = []
                    for response_data in individual_responses.values():
                        rating = response_data['rating']
                        ratings.append(rating)
                    
                    if len(ratings) > 1:
                        # Calculate variance (your existing code)
                        variance = np.var(ratings, ddof=1)
                        question_variances.append(variance)
                        
                        # Calculate entropy (your existing code)
                        rating_counts = Counter(ratings)
                        total = len(ratings)
                        entropy = 0
                        for count in rating_counts.values():
                            if count > 0:
                                prob = count / total
                                entropy -= prob * np.log2(prob)
                        question_entropies.append(entropy)
                        
                        # Calculate expert entropy for this question
                        expert_dist = row['expert_distribution']  # Already in your data!
                        expert_entropy = calculate_expert_entropy(expert_dist)
                        expert_entropies_for_model.append(expert_entropy)
                        all_expert_entropies.append(expert_entropy)
        
        # Store results (enhanced with expert entropy)
        if question_variances:
            results[model_name] = {
                'mean_variance': np.mean(question_variances),
                'std_variance': np.std(question_variances),
                'mean_entropy': np.mean(question_entropies),
                'std_entropy': np.std(question_entropies),
                'mean_expert_entropy': np.mean(expert_entropies_for_model),
                'std_expert_entropy': np.std(expert_entropies_for_model),
                'question_variances': question_variances,
                'question_entropies': question_entropies,
                'expert_entropies': expert_entropies_for_model,
                'n_questions': len(question_variances)
            }
    
    # Calculate overall expert entropy and std
    overall_expert_entropy = np.mean(all_expert_entropies)
    overall_expert_entropy_std = np.std(all_expert_entropies)
    overall_expert_entropy_error = overall_expert_entropy_std / np.sqrt(len(all_expert_entropies))
    
    # Create the comparison plot you requested
    plot_entropy_comparison_three_way(results, overall_expert_entropy, overall_expert_entropy_error, output_dir)
    
    # Your existing plots (unchanged)
    plot_sct_variance(results, output_dir)
    
    print(f"\nOverall Expert Entropy: {overall_expert_entropy:.4f} (±{overall_expert_entropy_std:.4f})")
    print(f"Based on {len(all_expert_entropies)} questions")
    
    return results

def plot_entropy_comparison_three_way(results, overall_expert_entropy, overall_expert_entropy_error, output_dir):
    """
    Create simple 3-bar comparison: Experts, CoT, Base (Experts first)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    model_names = list(results.keys())
    clean_labels = get_clean_labels(model_names)
    
    plt.figure(figsize=(10, 6))
    
    # Get the data for the model bars
    model_entropies = [results[model]['mean_entropy'] for model in model_names]
    model_entropy_errors = [results[model]['std_entropy'] / np.sqrt(results[model]['n_questions']) 
                           for model in model_names]
    
    # Prepare 3 values: Experts FIRST, then CoT, Base
    categories = ['Experts'] + clean_labels
    values = [overall_expert_entropy] + model_entropies
    errors = [overall_expert_entropy_error] + model_entropy_errors
    colors = ['darkred', 'mediumseagreen', 'orange']
    
    # Create the 3 bars
    bars = plt.bar(categories, values, yerr=errors, capsize=5, alpha=0.8, color=colors)
    
    plt.xlabel('Models', fontsize=14)
    plt.ylabel('Mean Response Entropy', fontsize=14)
    plt.title('SCT: Response Entropy Comparison', fontsize=16, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sct_entropy_three_way_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print comparison results
    print("\n" + "="*60)
    print("SCT ENTROPY THREE-WAY COMPARISON")
    print("="*60)
    print(f"Experts: {overall_expert_entropy:.4f} (±{overall_expert_entropy_error:.4f})")
    for i, model_name in enumerate(model_names):
        clean_name = clean_labels[i]
        r = results[model_name]
        print(f"{clean_name}: {r['mean_entropy']:.4f} (±{model_entropy_errors[i]:.4f})")
    print()

# ============================================================================
# REPLACE YOUR EXISTING FUNCTION CALL WITH THIS:
# ============================================================================

# Run the enhanced analysis (no metrics_dir needed!)
variance_output_dir = os.path.join(EVAL_PATH, 'plots', 'variance_analysis')
sct_variance_results = calculate_sct_variance_with_experts(processed_data, variance_output_dir)

# Run the analysis
#variance_output_dir = os.path.join(EVAL_PATH, 'plots', 'variance_analysis')
#sct_variance_results = calculate_sct_variance(processed_data, variance_output_dir)

In [ ]:
def generate_sct_summary_report(model_data_dict, output_dir):
    """
    Generate a comprehensive summary report of SCT performance.
    """
    report_data = []
    
    for model_name, df_eval in model_data_dict.items():
        metrics = calculate_overall_sct_metrics(df_eval)
        metrics['model_name'] = model_name
        report_data.append(metrics)
    
    # Create summary DataFrame
    summary_df = pd.DataFrame(report_data)
    
    # Save to CSV
    summary_df.to_csv(os.path.join(output_dir, 'sct_summary_report.csv'), index=False)
    
    # Print formatted report
    print("\n" + "="*80)
    print("SCT EVALUATION SUMMARY REPORT")
    print("="*80)
    
    for _, row in summary_df.iterrows():
        print(f"\nModel: {row['model_name']}")
        print(f"  Overall SCT Score: {row['overall_sct_score']:.3f} ± {row['score_std']:.3f}")
        print(f"  Self-Consistency: {row['overall_self_consistency']:.3f}")
        print(f"  Expert Modal Agreement: {row['expert_modal_agreement']:.3f}")
        print(f"  Questions Evaluated: {row['total_questions']}")
        if row['average_uncertainty'] is not None:  # Changed from confidence
            print(f"  Average Uncertainty: {row['average_uncertainty']:.1f}")
        if row['average_reasoning_tokens'] is not None:
            print(f"  Average Reasoning Tokens: {row['average_reasoning_tokens']:.0f}")
    
    return summary_df

# Generate the summary report
if processed_data:
    summary_df = generate_sct_summary_report(processed_data, EVAL_PATH)
    print(f"\nSummary report saved to: {os.path.join(EVAL_PATH, 'sct_summary_report.csv')}")
else:
    print("No processed data available for summary report")